In [1]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub
try:
    import huggingface_hub
except ImportError:
    !pip install huggingface_hub
try:
    import ipywidgets
except ImportError:
    !pip install ipywidgets
    !pip install --upgrade ipywidgets

!pip install --upgrade notebook jupyterlab

from src.dataset.DroneSegDataSet import MyDataset
from src.logging.Logger import Logger
from src.model.DroneSegModel import DroneSegModel
from src.training.TrainPhase import train_phase
from src.dataset.CheckDataset import check_dataset

当前脚本路径: F:\2025ProjectDroneSeg\2026.3.25_BBC6521_DroneSegModel\src\dataset
项目根路径: F:\2025ProjectDroneSeg\2026.3.25_BBC6521_DroneSegModel


In [ ]:
# 检查模型消耗的显存
from src.model.DroneSegModel import main

main()


In [ ]:
def print_model_params(model):
    print("Model parameter summary:")
    print("-" * 60)
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            numel = param.numel()
            total += numel
            print(f"{name:<50} {numel:>12,}")
    print("-" * 60)
    print(f"{'Total trainable parameters':<50} {total:>12,}")


In [2]:
# 检查硬件环境
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from torchvision import transforms
# 加载数据集
ds_path = check_dataset()
dataset = MyDataset(
    'drone_seg_dataset/classes_dataset/classes_dataset/original_images',
    'drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
    transform=transforms.Compose([
        transforms.ToTensor(),
    ])
)

from torch.utils.data import random_split, DataLoader
import numpy as np

train_set, test_set = random_split(dataset, [int(0.9 * len(dataset)), len(dataset) - int(0.9 * len(dataset))])
train_loader = DataLoader(dataset=train_set, batch_size=2, shuffle=True)
test_loader = DataLoader(dataset=test_set, batch_size=2, shuffle=False)

model = DroneSegModel(max_channels=64).to(device)

print(f"数据集总样本数: {len(dataset)}")
print(f"训练集样本数: {len(train_set)}")
print(f"测试集样本数: {len(test_set)}")

img0, lbl0, _ = train_set.__getitem__(0)
print(f"Train set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
lbl_unique = torch.unique(lbl0)
print(f"Train set - unique labels in lbl0: {lbl_unique}")

img0, lbl0, _ = test_set.__getitem__(0)
print(f"Test set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
lbl_unique = torch.unique(lbl0)
print(f"Train set - unique labels in lbl0: {lbl_unique}")

img0, lbl0, _ = dataset.__getitem__(0)

print("图像1的形状:", img0.shape)
print("标签1的形状:", lbl0.shape)

img0_batched = img0.unsqueeze(0)
ans0 = model(img0_batched.to(device), mode="pretrain")
print("预训练模式模型输出的形状:", ans0.shape)

img0_batched = img0.unsqueeze(0)
ans0 = model(img0_batched.to(device), mode="train")
print("训练模式模型输出的形状:", ans0.shape)

sample_output = torch.argmax(ans0, dim=1).detach().cpu().numpy()
print("模型输出的唯一值 (类别索引):", np.unique(sample_output))

print("Train logger: Unet_Segmentation_Logger")
logger = Logger(
    title='train_model',
    log_format=[
        'phase', 'epoch', 'elapsed_time', 'loss', 'miou', 'accuracy', 'time'
    ],
    output_frequency=1,
)

print("model parameters count")
print_model_params(model)

# print("Model summary: ")
# print(model)


Using device: cuda
Dataset already exists.
成功匹配文件: 489.png, 386.png, 419.png 等 400 个文件
输入通道数表 (in_c):
层 0: [32, 48, 104, 232, 256, 256, 256]
层 1: [16, 56, 132, 328, 384, 384, 384]
层 2: [8, 28, 96, 264, 384, 384, 384]
层 3: [4, 12, 40, 136, 256, 256, 256]

输出通道数表 (out_c):
层 0: [32, 48, 104, 128, 128, 128, 128]
层 1: [16, 56, 128, 128, 128, 128, 128]
层 2: [8, 28, 96, 128, 128, 128, 128]
层 3: [4, 12, 40, 128, 128, 128, 128]
数据集总样本数: 400
训练集样本数: 360
测试集样本数: 40
提取的特征维度:  torch.Size([22, 736, 960])
Train set - img shape: torch.Size([22, 736, 960]), lbl shape: torch.Size([1, 736, 960])
Train set - unique labels in lbl0: tensor([0, 1, 2, 3, 4])
提取的特征维度:  torch.Size([22, 736, 960])
Test set - img shape: torch.Size([22, 736, 960]), lbl shape: torch.Size([1, 736, 960])
Train set - unique labels in lbl0: tensor([0, 2, 3, 4])
提取的特征维度:  torch.Size([22, 736, 960])
图像1的形状: torch.Size([22, 736, 960])
标签1的形状: torch.Size([1, 736, 960])
预训练模式模型输出的形状: torch.Size([1, 5, 736, 960])


OutOfMemoryError: CUDA out of memory. Tried to allocate 88.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 12.16 GiB is allocated by PyTorch, and 215.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def train_all_phases(
        model,
        dataloader,
        criterion,
        train_config,
        logger,
        device,
):

    for phase_idx, (config) in enumerate(train_config):

        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config['lr'],
            momentum=config['momentum'],
        )

        print(f"Phase {phase_idx} Starting | type: {config['type']}, lr: {config['lr']}, momentum: {config['momentum']}, epochs: {config['epochs']}")
        logger.start_new_phase(f"Phase {phase_idx} - {config['type']}")

        train_phase(
            model=model,
            dataloader=dataloader,
            logger=logger,
            epochs=config['epochs'],
            is_pretrain=(config['type'] == 'pretrain'),
            sample_period=min(50, config['epochs'] // 2),  # 每个阶段至少保存 2 次样本
            device=device,
            logging_info={
                'phase': phase_idx,
            },
            criterion=criterion,
            optimizer=optimizer,
        )

        logger.end_current_phase(model_to_save=model)

    logger.finalize_and_plot_all()
    torch.cuda.empty_cache()


In [ ]:
INIT_MODEL_PATH = "" # 在此处填入.pth文件路径以加载预训练权重，留空则从头开始训练
if INIT_MODEL_PATH:
    model.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=device))
    print(f"已从 '{INIT_MODEL_PATH}' 加载模型参数。")
else:
    print("未指定预训练模型，从头开始训练。")

# --- 必须定义类别数量 ---
NUM_CLASSES = 5 # 请将此值替换为您数据集的实际类别数量，例如PASCAL VOC有21类

# --- 定义损失函数和优化器 ---
criterion = torch.nn.CrossEntropyLoss()
# 优化器使用 SGD，学习率与动量动态调整
train_config = [
    # --- Pretrain 阶段: 渐进式预热 ---
    # 第1阶段: 高学习率，快速探索，较低动量
    {'lr': 0.2, 'momentum': 0.85, 'epochs': 40, 'type': 'pretrain'},
    # 第2阶段: 中等学习率，稳定预训练
    {'lr': 0.1, 'momentum': 0.9, 'epochs': 40, 'type': 'pretrain'},
    # 第3阶段: 较低学习率，精细化预训练
    {'lr': 0.05, 'momentum': 0.9, 'epochs': 40, 'type': 'pretrain'},

    # --- Train 阶段: 精细收敛 ---
    # 第4阶段: 标准学习率开始精调
    {'lr': 0.01, 'momentum': 0.9, 'epochs': 50, 'type': 'train'},
    # 第5阶段: 学习率衰减，进一步收敛
    {'lr': 0.005, 'momentum': 0.9, 'epochs': 30, 'type': 'train'},
    # 第6阶段: 低学习率扫尾，提升精度
    {'lr': 0.001, 'momentum': 0.95, 'epochs': 25, 'type': 'train'},
]

train_all_phases(
    model=model,
    dataloader=train_loader,
    criterion=criterion,
    train_config=train_config,
    logger=logger,
    device=device,
)
